# Notebook 01 - Data Generation

**Project:** Mirae Asset Digital Platform - User Analytics  
**Phase covered:** Phase 1 - Synthetic Raw Data Generation  
**Objective:** Generate the synthetic source data used by the full analytics project: visitors, registered users, sessions, valid post-signup transactions, causal funnel events, and marketing campaigns.

**Generation principles:**
- Every behavioural record is tied to an existing `user_id`.
- Sessions, events, and transactions occur on or after the user's signup date or first observed visitor date.
- Visitor-only users are included so the Visit -> Signup funnel has real top-of-funnel drop-off.
- Registered users follow a cumulative chronological funnel: signup -> add_to_cart -> purchase.
- Paid campaign channels are named so they can be mapped cleanly to acquisition channels in later CAC/ROI analysis.
- The raw CSVs remain synthetic and safe to include in the repository for dashboard deployment.


In [1]:
## Import Libraries
import pandas as pd
import numpy as np
from pathlib import Path

np.random.seed(42)

START_DATE = pd.Timestamp('2023-01-01')
END_DATE = pd.Timestamp('2023-06-29')
N_DAYS = (END_DATE - START_DATE).days + 1


In [2]:
## Generate Users / Visitor Dataset
n_registered_users = 10000
visitor_to_signup_rate = 0.80
n_visitors = int(n_registered_users / visitor_to_signup_rate)
n_anonymous_visitors = n_visitors - n_registered_users
n_users = n_registered_users  # registered analytical users used by downstream notebooks

states = [
    'Odisha', 'Maharashtra', 'Karnataka', 'Delhi', 'West Bengal',
    'Tamil Nadu', 'Uttar Pradesh', 'Gujarat', 'Rajasthan', 'Telangana'
]

registered_users = pd.DataFrame({
    'user_id': range(1, n_registered_users + 1),
    'signup_date': START_DATE + pd.to_timedelta(np.random.randint(0, N_DAYS, n_registered_users), unit='D'),
    'country': 'India',
    'state': np.random.choice(states, n_registered_users),
    'device': np.random.choice(['Mobile', 'Desktop'], n_registered_users, p=[0.7, 0.3]),
    'age': np.random.randint(18, 60, n_registered_users),
    'gender': np.random.choice(['Male', 'Female'], n_registered_users),
    'acquisition_channel': np.random.choice(
        ['Facebook Ads', 'Google Ads', 'Organic', 'Referral'],
        n_registered_users,
        p=[0.26, 0.25, 0.25, 0.24]
    ),
    'is_registered': 1,
})

anonymous_visitors = pd.DataFrame({
    'user_id': range(n_registered_users + 1, n_visitors + 1),
    # For visitor-only rows this stores first observed visit date, not account creation.
    'signup_date': START_DATE + pd.to_timedelta(np.random.randint(0, N_DAYS, n_anonymous_visitors), unit='D'),
    'country': 'India',
    'state': np.random.choice(states, n_anonymous_visitors),
    'device': np.random.choice(['Mobile', 'Desktop'], n_anonymous_visitors, p=[0.72, 0.28]),
    'age': np.random.randint(18, 60, n_anonymous_visitors),
    'gender': np.random.choice(['Male', 'Female'], n_anonymous_visitors),
    'acquisition_channel': np.random.choice(
        ['Facebook Ads', 'Google Ads', 'Organic', 'Referral'],
        n_anonymous_visitors,
        p=[0.28, 0.26, 0.24, 0.22]
    ),
    'is_registered': 0,
})

users = pd.concat([registered_users, anonymous_visitors], ignore_index=True)

print(f'Top-of-funnel visitors : {len(users):,}')
print(f'Registered users       : {registered_users["is_registered"].sum():,}')
print(f'Visitor -> signup rate : {len(registered_users) / len(users):.1%}')
users.head()


Top-of-funnel visitors : 12,500
Registered users       : 10,000
Visitor -> signup rate : 80.0%


,user_id,signup_date,country,state,device,age,gender,acquisition_channel,is_registered
0,1,2023-04-13,India,Uttar Pradesh,Mobile,45,Male,Facebook Ads,1
1,2,2023-06-29,India,Karnataka,Mobile,47,Male,Referral,1
2,3,2023-04-03,India,Gujarat,Mobile,58,Male,Organic,1
3,4,2023-01-15,India,Maharashtra,Mobile,46,Female,Facebook Ads,1
4,5,2023-04-17,India,Maharashtra,Mobile,51,Female,Google Ads,1


In [3]:
## Generate Sessions Dataset
n_sessions = 50000

registered_user_ids = registered_users['user_id'].to_numpy()
session_user_ids = np.random.choice(registered_user_ids, n_sessions)
session_base = pd.DataFrame({'user_id': session_user_ids}).merge(
    registered_users[['user_id', 'signup_date']], on='user_id', how='left'
)
session_windows = (END_DATE - session_base['signup_date']).dt.days + 1
session_offsets = [np.random.randint(0, max(int(window), 1)) for window in session_windows]

sessions = pd.DataFrame({
    'session_id': range(1, n_sessions + 1),
    'user_id': session_user_ids,
    'session_date': session_base['signup_date'] + pd.to_timedelta(session_offsets, unit='D'),
    'duration_minutes': np.random.randint(1, 60, n_sessions),
    'pages_viewed': np.random.randint(1, 15, n_sessions)
})

sessions.head()


,session_id,user_id,session_date,duration_minutes,pages_viewed
0,1,7359,2023-05-20,20,12
1,2,6719,2023-03-12,19,6
2,3,2825,2023-05-31,20,13
3,4,5204,2023-04-21,16,9
4,5,3549,2023-06-25,40,3


In [4]:
## Generate Transactions Dataset
n_transactions = 15000
buyer_rate = 0.48

registered_user_ids = registered_users['user_id'].to_numpy()
buyer_ids = np.random.choice(
    registered_user_ids,
    size=int(n_registered_users * buyer_rate),
    replace=False
)
remaining_transactions = n_transactions - len(buyer_ids)
transaction_user_ids = np.concatenate([
    buyer_ids,
    np.random.choice(buyer_ids, remaining_transactions, replace=True)
])
np.random.shuffle(transaction_user_ids)
transaction_base = pd.DataFrame({'user_id': transaction_user_ids}).merge(
    registered_users[['user_id', 'signup_date']], on='user_id', how='left'
)
transaction_windows = (END_DATE - transaction_base['signup_date']).dt.days + 1
transaction_offsets = [np.random.randint(0, max(int(window), 1)) for window in transaction_windows]

transactions = pd.DataFrame({
    'transaction_id': range(1, n_transactions + 1),
    'user_id': transaction_user_ids,
    'amount': np.random.randint(100, 5000, n_transactions),
    'transaction_date': transaction_base['signup_date'] + pd.to_timedelta(transaction_offsets, unit='D'),
    'payment_method': np.random.choice(
        ['UPI', 'Credit Card', 'Debit Card', 'Net Banking'], n_transactions
    )
})

transactions.head()


,transaction_id,user_id,amount,transaction_date,payment_method
0,1,5347,3720,2023-06-02,UPI
1,2,5569,4941,2023-04-17,UPI
2,3,8521,1148,2023-06-28,Net Banking
3,4,1571,895,2023-02-18,Credit Card
4,5,3015,4708,2023-06-19,UPI


In [5]:
## Generate Causal Funnel Events Dataset
# The event table includes visitor-only rows so Visit -> Signup has real top-of-funnel drop-off.
# Registered users then follow a cumulative path: signup -> add_to_cart -> purchase.
target_events = 90000
cart_rate = 0.70

registered_id_set = set(map(int, registered_users['user_id']))
visitor_id_set = set(map(int, users['user_id']))
buyer_id_set = set(map(int, buyer_ids))
nonbuyer_ids = registered_users.loc[~registered_users['user_id'].isin(buyer_id_set), 'user_id'].to_numpy()
cart_target = int(n_registered_users * cart_rate)
cart_nonbuyer_count = max(cart_target - len(buyer_id_set), 0)
cart_nonbuyer_ids = set(map(int, np.random.choice(nonbuyer_ids, cart_nonbuyer_count, replace=False)))
add_to_cart_ids = buyer_id_set | cart_nonbuyer_ids

signup_lookup = users.set_index('user_id')['signup_date'].to_dict()
first_purchase_lookup = transactions.groupby('user_id')['transaction_date'].min().to_dict()
txn_dates_by_user = transactions.groupby('user_id')['transaction_date'].apply(list).to_dict()

def random_date_between(start, end):
    days = max((end - start).days, 0)
    return start + pd.to_timedelta(np.random.randint(0, days + 1), unit='D')

records = []
event_id = 1
first_cart_lookup = {}

for row in users[['user_id', 'signup_date', 'is_registered']].itertuples(index=False):
    uid = int(row.user_id)
    first_seen_date = row.signup_date

    records.append((event_id, uid, 'visit', first_seen_date)); event_id += 1

    if int(row.is_registered) == 1:
        records.append((event_id, uid, 'signup', first_seen_date)); event_id += 1

        if uid in add_to_cart_ids:
            cart_end = first_purchase_lookup.get(uid, END_DATE)
            cart_date = random_date_between(first_seen_date, cart_end)
            first_cart_lookup[uid] = cart_date
            records.append((event_id, uid, 'add_to_cart', cart_date)); event_id += 1

        if uid in buyer_id_set:
            purchase_date = first_purchase_lookup[uid]
            records.append((event_id, uid, 'purchase', purchase_date)); event_id += 1

extra_events = target_events - len(records)
extra_user_ids = np.random.choice(users['user_id'], extra_events, replace=True)

for uid_raw in extra_user_ids:
    uid = int(uid_raw)
    eligible = ['visit']
    probs = [1.0]
    if uid in add_to_cart_ids and uid in buyer_id_set:
        eligible = ['visit', 'add_to_cart', 'purchase']
        probs = [0.55, 0.25, 0.20]
    elif uid in add_to_cart_ids:
        eligible = ['visit', 'add_to_cart']
        probs = [0.75, 0.25]

    event_type = np.random.choice(eligible, p=probs)
    first_seen_date = signup_lookup[uid]
    if event_type == 'purchase':
        event_date = np.random.choice(txn_dates_by_user[uid])
    elif event_type == 'add_to_cart':
        event_date = random_date_between(first_cart_lookup.get(uid, first_seen_date), END_DATE)
    else:
        event_date = random_date_between(first_seen_date, END_DATE)

    records.append((event_id, uid, event_type, event_date)); event_id += 1

events = pd.DataFrame(records, columns=['event_id', 'user_id', 'event_type', 'event_date'])
events = events.sort_values(['user_id', 'event_date', 'event_id']).reset_index(drop=True)

events.head()


,event_id,user_id,event_type,event_date
0,1,1,visit,2023-04-13
1,2,1,signup,2023-04-13
2,65063,1,visit,2023-04-19
3,80363,1,visit,2023-05-08
4,89167,1,visit,2023-05-24


In [6]:
## Generate Marketing Campaign Dataset
n_campaigns = 50

campaigns = pd.DataFrame({
    'campaign_id': range(1, n_campaigns + 1),
    'channel': np.random.choice(
        ['Facebook Ads', 'Google Ads', 'Referral', 'Organic'],
        n_campaigns,
        p=[0.30, 0.30, 0.25, 0.15]
    ),
    'cost': np.random.randint(5000, 50000, n_campaigns),
    'start_date': START_DATE + pd.to_timedelta(np.random.randint(0, 150, n_campaigns), unit='D')
})

# Organic activity is tracked as campaigns/content pushes, but paid media spend is zero.
campaigns.loc[campaigns['channel'].eq('Organic'), 'cost'] = 0

campaigns.head()


,campaign_id,channel,cost,start_date
0,1,Google Ads,28479,2023-05-23
1,2,Google Ads,16506,2023-05-20
2,3,Google Ads,23412,2023-03-02
3,4,Facebook Ads,32748,2023-01-14
4,5,Referral,12945,2023-01-13


In [7]:
## Save Raw CSV Files
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = BASE_DIR / 'data' / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

users.to_csv(RAW_DIR / 'users.csv', index=False)
sessions.to_csv(RAW_DIR / 'sessions.csv', index=False)
transactions.to_csv(RAW_DIR / 'transactions.csv', index=False)
events.to_csv(RAW_DIR / 'events.csv', index=False)
campaigns.to_csv(RAW_DIR / 'campaigns.csv', index=False)

print('All raw files saved.')
print(f'Visitors    : {users.shape}  (registered={int(users["is_registered"].sum()):,})')
print(f'Sessions    : {sessions.shape}')
print(f'Transactions: {transactions.shape}')
print(f'Events      : {events.shape}')
print(f'Campaigns   : {campaigns.shape}')


All raw files saved.
Visitors    : (12500, 9)  (registered=10,000)
Sessions    : (50000, 5)
Transactions: (15000, 5)
Events      : (90000, 4)
Campaigns   : (50, 4)


In [8]:
## Validation Checks
users_lookup = users[['user_id', 'signup_date']]

session_check = sessions.merge(users_lookup, on='user_id', how='left')
transaction_check = transactions.merge(users_lookup, on='user_id', how='left')
event_check = events.merge(users_lookup, on='user_id', how='left')

assert session_check['signup_date'].notna().all()
assert transaction_check['signup_date'].notna().all()
assert event_check['signup_date'].notna().all()
assert (session_check['session_date'] >= session_check['signup_date']).all()
assert (transaction_check['transaction_date'] >= transaction_check['signup_date']).all()
assert (event_check['event_date'] >= event_check['signup_date']).all()

funnel_first = events.pivot_table(index='user_id', columns='event_type', values='event_date', aggfunc='min')
required_stages = ['visit', 'signup', 'add_to_cart', 'purchase']
assert all(stage in funnel_first.columns for stage in required_stages)
assert funnel_first['visit'].notna().sum() > funnel_first['signup'].notna().sum()
assert funnel_first['signup'].notna().sum() >= funnel_first['add_to_cart'].notna().sum()
assert funnel_first['add_to_cart'].notna().sum() >= funnel_first['purchase'].notna().sum()
assert (funnel_first.dropna(subset=['signup', 'visit'])['signup'] >= funnel_first.dropna(subset=['signup', 'visit'])['visit']).all()
assert (funnel_first.dropna(subset=['add_to_cart', 'signup'])['add_to_cart'] >= funnel_first.dropna(subset=['add_to_cart', 'signup'])['signup']).all()
assert (funnel_first.dropna(subset=['purchase', 'add_to_cart'])['purchase'] >= funnel_first.dropna(subset=['purchase', 'add_to_cart'])['add_to_cart']).all()

print('Validation passed: behavioural records are tied to visitor/user IDs and occur on or after first observed date.')
print('Funnel validation passed: visit -> signup -> add_to_cart -> purchase is cumulative and chronological.')
print(f'Buyer conversion in raw transactions: {transactions["user_id"].nunique() / len(registered_users):.1%}')
print(f'Total transaction revenue: Rs {transactions["amount"].sum():,.0f}')
print(f'Funnel unique users: ' + ' -> '.join(f'{stage}={funnel_first[stage].notna().sum():,}' for stage in required_stages))


Validation passed: behavioural records are tied to visitor/user IDs and occur on or after first observed date.
Funnel validation passed: visit -> signup -> add_to_cart -> purchase is cumulative and chronological.
Buyer conversion in raw transactions: 48.0%
Total transaction revenue: Rs 38,095,829
Funnel unique users: visit=12,500 -> signup=10,000 -> add_to_cart=7,000 -> purchase=4,800
